In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install tensorflow pandas scikit-learn joblib matplotlib


Mounted at /content/drive


In [ ]:
import pandas as pd

csv_path = "/content/drive/MyDrive/Research_code/data/tabular/oral_cancer_prediction_dataset.csv"
df = pd.read_csv(csv_path)
df.head()


,ID,Country,Age,Gender,Tobacco Use,Alcohol Consumption,HPV Infection,Betel Quid Use,Chronic Sun Exposure,Poor Oral Hygiene,...,Difficulty Swallowing,White or Red Patches in Mouth,Tumor Size (cm),Cancer Stage,Treatment Type,"Survival Rate (5-Year, %)",Cost of Treatment (USD),Economic Burden (Lost Workdays per Year),Early Diagnosis,Oral Cancer (Diagnosis)
0,1,Italy,36,Female,Yes,Yes,Yes,No,No,Yes,...,No,No,0.000000,0,No Treatment,100.000000,0.00,0,No,No
1,2,Japan,64,Male,Yes,Yes,Yes,No,Yes,Yes,...,No,No,1.782186,1,No Treatment,83.340103,77772.50,177,No,Yes
2,3,UK,37,Female,No,Yes,No,No,Yes,Yes,...,No,Yes,3.523895,2,Surgery,63.222871,101164.50,130,Yes,Yes
3,4,Sri Lanka,55,Male,Yes,Yes,No,Yes,No,Yes,...,No,No,0.000000,0,No Treatment,100.000000,0.00,0,Yes,No
4,5,South Africa,68,Male,No,No,No,No,No,Yes,...,No,No,2.834789,3,No Treatment,44.293199,45354.75,52,No,Yes


In [ ]:
# Handle missing values & prepare data

# Replace Yes/No with 1/0
yes_no_columns = [
    'Tobacco Use', 'Alcohol Consumption', 'HPV Infection', 'Betel Quid Use',
    'Chronic Sun Exposure', 'Poor Oral Hygiene', 'Family History of Cancer',
    'Compromised Immune System', 'Oral Lesions', 'Unexplained Bleeding',
    'Difficulty Swallowing', 'White or Red Patches in Mouth', 'Early Diagnosis',
    'Oral Cancer (Diagnosis)'   # Target
]

for col in yes_no_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# One‑hot encode categorical column
df = pd.get_dummies(df, columns=['Gender', 'Country', 'Diet (Fruits & Vegetables Intake)', 'Treatment Type'], drop_first=True)

# Drop ID column if exists
if 'ID' in df.columns:
    df.drop(columns=['ID'], inplace=True)

df.head()


,Age,Tobacco Use,Alcohol Consumption,HPV Infection,Betel Quid Use,Chronic Sun Exposure,Poor Oral Hygiene,Family History of Cancer,Compromised Immune System,Oral Lesions,...,Country_Sri Lanka,Country_Taiwan,Country_UK,Country_USA,Diet (Fruits & Vegetables Intake)_Low,Diet (Fruits & Vegetables Intake)_Moderate,Treatment Type_No Treatment,Treatment Type_Radiation,Treatment Type_Surgery,Treatment Type_Targeted Therapy
0,36,1,1,1,0,0,1,0,0,0,...,False,False,False,False,True,False,True,False,False,False
1,64,1,1,1,0,1,1,0,0,0,...,False,False,False,False,False,False,True,False,False,False
2,37,0,1,0,0,1,1,0,0,0,...,False,False,True,False,False,True,False,False,True,False
3,55,1,1,0,1,0,1,0,0,1,...,True,False,False,False,False,True,True,False,False,False
4,68,0,0,0,0,0,1,0,0,0,...,False,False,False,False,False,False,True,False,False,False


In [ ]:
# Train Test Split
X = df.drop(columns=['Oral Cancer (Diagnosis)'])
y = df['Oral Cancer (Diagnosis)']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape


((67937, 42), (16985, 42))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

print("Model training completed!")


Model training completed!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      8560
           1       1.00      1.00      1.00      8425

    accuracy                           1.00     16985
   macro avg       1.00      1.00      1.00     16985
weighted avg       1.00      1.00      1.00     16985



In [ ]:
#Cell 7
import joblib

joblib.dump(model, "/content/model_tabular.pkl")
joblib.dump(X.columns, "/content/tabular_features.pkl")

print("Model Saved!")


Model Saved!


In [ ]:
import os

folder = "/content/drive/MyDrive/Research_code"
print(os.listdir(folder))


['data', 'image_model.h5', 'image_feature_extractor.h5', 'preprocessor_tabular.joblib', 'tabular_encoder.h5', 'tabular_model.h5', 'image_feature_extractor.keras', 'train_img_features.npy', 'val_img_features.npy', 'train_img_labels.npy', 'val_img_labels.npy', 'image_model.keras']


In [ ]:

# Cell 8 – download correct saved files
from google.colab import files

# Download tabular model
files.download("/content/drive/MyDrive/Research_code/tabular_model.h5")

# Download preprocessor (NOTE: correct name)
files.download("/content/drive/MyDrive/Research_code/preprocessor_tabular.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#cell 9
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_img_gen = train_datagen.flow_from_directory(
    "/content/drive/MyDrive/Research_code/data/images/train",
    target_size=(224, 224),
    batch_size=64,
    class_mode="binary",
    shuffle=True
)

val_img_gen = val_test_datagen.flow_from_directory(
    "/content/drive/MyDrive/Research_code/data/images/val",
    target_size=(224, 224),
    batch_size=64,
    class_mode="binary",
    shuffle=True
)


Found 750 images belonging to 2 classes.
Found 0 images belonging to 0 classes.


In [ ]:
# Cell 10 — Fine-tuned EfficientNet

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

# 🔓 Unfreeze top layers
for layer in base_model.layers[-40:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid')(x)

image_model = Model(inputs=base_model.input, outputs=output)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
#Cell 11
from tensorflow.keras.optimizers import Adam

image_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
#callback, run before cell 12
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6
)


In [ ]:
import tensorflow as tf
import math

train_ds = tf.data.Dataset.from_generator(
    lambda: train_img_gen,
    output_signature=(
        tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32),
    )
)

val_ds = tf.data.Dataset.from_generator(
    lambda: val_img_gen,
    output_signature=(
        tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32),
    )
)

steps_per_epoch = math.ceil(train_img_gen.samples / train_img_gen.batch_size)
validation_steps = math.ceil(val_img_gen.samples / val_img_gen.batch_size)

history_img = image_model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)


Epoch 1/20
10/12 ━━━━━━━━━━━━━━━━━━━━ 31s 16s/step - accuracy: 0.6007 - loss: 0.6581

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 20s/step - accuracy: 0.6142 - loss: 0.6495 

In [ ]:
#cell 13
from tensorflow.keras.models import load_model
import numpy as np

# Load saved models
image_model = load_model("/content/drive/MyDrive/Research_code/image_model.keras")
feat_extractor = load_model("/content/drive/MyDrive/Research_code/image_feature_extractor.keras")

print("Loaded image model and feature extractor successfully!")


In [ ]:
#cell 14
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)

test_gen = ImageDataGenerator(rescale=1./255)

test_img_gen = test_gen.flow_from_directory(
    "/content/drive/MyDrive/Research_code/data/images/test",
    target_size=IMG_SIZE,
    batch_size=16,
    class_mode='binary',
    shuffle=False
)


In [ ]:
#cell 15
test_loss, test_acc = image_model.evaluate(test_img_gen)
print("📌 Image Model Accuracy on Test Set:", round(test_acc * 100, 2), "%")


In [ ]:
import os

base = "/content/drive/MyDrive/Research_code/data/images"
for split in ["train", "test"]:
    print("\n📂", split.upper())
    split_path = os.path.join(base, split)

    if not os.path.exists(split_path):
        print(f"❌ Folder not found: {split_path}")
        continue

    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            print(f"{cls}: {len(os.listdir(cls_path))} images")


In [ ]:
#Cell‑16 — Extract Image Features
import numpy as np

# Reload feature extractor
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224,224)

base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224,224,3))
base.trainable = False
feature_model = Model(inputs=base.input, outputs=GlobalAveragePooling2D()(base.output))

test_data = ImageDataGenerator(rescale=1./255).flow_from_directory(
    "/content/drive/MyDrive/Research_code/data/images/test",
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

img_features = feature_model.predict(test_data)
np.save("test_img_features.npy", img_features)
np.save("test_img_labels.npy", test_data.classes)

print("Saved extracted image features 🎉")
